# DS4DS Exercise Sheet 8

**General Instructions:**

- Review the weekly course material (lectures, readings, slides, etc.) before starting the exercises.
- Complete all assigned exercises independently before the Q&A session.
- Please use Julia Version 1.12.x to ensure compatibility.
- Please only write between the `#--- YOUR CODE STARTS HERE ---#` and `#--- YOUR CODE ENDS HERE ---#` comments.

## Exercise 1 - Automatic and Approximate Derivatives

As some of you might have noticed with the last exercise sheet, 
manual differentiation of complicated functions tends to be error-prone and requires
knowledge of the inner workings of said functions.
Luckily, there are ways to avoid hand-crafted derivatives.
If you have Julia functions performing tedious computations, you can use one of the many
*Automatic Differentiation*  (AD) packages.
A good AD package gives exact derivatives for differentiable functions, and oftentimes they 
employ clever tricks to do so fast and in a memory-efficient way.
In case your function relies on external software (e.g., simulations), you could try to
approximate the gradients.
Traditionally, *Finite Differences* are the go-to approach for gradient approximations.

Automatic Differentiation (AD): This applies the chain rule algorithmically to code, yielding exact derivatives. It is highly efficient and the preferred method for differentiable functions written entirely in your programming language.

Finite Differences: When a function relies on external software (like an external simulation solver where the internal equations are a "black box"), you cannot use AD. Instead, you must approximate the gradients by observing how the output changes when you slightly tweak the inputs.

In this exercise, we are going to implement a finite difference scheme for multi-variate functions
ourselves.
Moreover, we take a look at the AD packages `FineteDiff`, `ForwardDiff` and `Zygote`.

### Manual Finite Differences


We are interested in the (first-order) partial derivatives of $\mathcal L\colon ℝ^q \to ℝ.$

The [finite difference operator](https://en.wikipedia.org/wiki/Finite_difference) $Δ_h^{w_i}$ 
with grid-size $h > 0$ maps $\mathcal L$ to $Δ_h^{w_i}[\mathcal L]$, and we would like 
$Δ_h^{w_i}[\mathcal L](\mathbf w) \approx ∂_{w_i} \mathcal L(\mathbf w)$.
That is, the operator should approximate the partial derivative of $\mathcal L$ with respect to $w_i$ 
at $\mathbf w \in ℝ^q$.
If $\mathcal L$ is two-times continuously differentiable, then the central finite difference scheme 
is of order 2, meaning that the error of the first-order derivatives is $\mathcal O(h^2)$.

There are, of course, alternatives: forward and backward schemes, and schemes of higher 
accuracy or order, see [this table in the Wikipedia](https://en.wikipedia.org/wiki/Finite_difference_coefficient).
For now, we want to stick with the central finite diffence scheme:
$$
Δ_h^{w_i}[\mathcal L](\mathbf w) = \frac{\mathcal L(\mathbf w + h \mathbf e_i) - \mathcal L(\mathbf w - h \mathbf e_i)}{2 h}.
$$
The $i^{\text{th}}$ unit vector $\mathbf e_i$ is all zeros, except at position $i$, where it has entry $1$.

You have a function $\mathcal{L}$ that takes a vector of $q$ variables and outputs a single scalar value (e.g., $\mathcal{L}: \mathbb{R}^q \rightarrow \mathbb{R}$). You want to find the first-order partial derivative with respect to a specific variable, $w_i$.Instead of an exact analytical derivative, you use the finite difference operator $\Delta_h^{w_i}$ to approximate it using a very small step size, $h$.

Here is how the components break down:

$\mathbf{w}$: Your current input vector.

$\mathbf{e}_i$: The $i^{\text{th}}$ unit vector. This vector is entirely zeros except for a $1$ at the $i^{\text{th}}$ position. Multiplying by $h\mathbf{e}_i$ ensures you are only perturbing the single variable $w_i$ by the step size $h$, leaving all other variables constant.

The Numerator: You evaluate the function slightly forward ($\mathbf{w} + h\mathbf{e}_i$) and slightly backward ($\mathbf{w} - h\mathbf{e}_i$), then find the difference.

The Denominator: Because you stepped forward by $h$ and backward by $h$, the total distance between the two evaluation points is $2h$.

### Exercise 1.a)
Write a function `finite_diff_grad` that takes a function `func` ($\mathcal L$), 
an input vector `w` and 
a scalar grid-size `h`, and returns the central finite difference gradient approximation 
$Δ_h[\mathcal L](\mathbf w) = [Δ_h^{w_1}[\mathcal L](\mathbf w), \ldots, Δ_h^{w_q}[\mathcal L](\mathbf w)]^T \in ℝ^q$. \
Do so by completing the cell below:

In [9]:
"""
    finite_diff_grad(func, w::AbstractVector{W}, h::H=0.0001f0) where {W<:Real, H<:Real}

Return the finite difference gradient approximation of `func` at input `w`.
"""
import numpy as np

def finite_diff_grad(func, w, h=0.0001):
    """
    Return the finite difference gradient approximation of `func` at input `w`.
    """
    assert h > 0, "Grid-size must be positive."
    
    # Ensure w is a float array to avoid integer truncation during perturbation
    w = np.array(w, dtype=float)
    
    # Pre-allocate solution array (equivalent to `zeros(length(w))` in Julia)
    dw_func = np.zeros_like(w)
    
    # Fill `dw_func` to contain the finite difference approximations
    for i in range(len(w)):
        # Create separate copies for the forward and backward steps
        w_forward = w.copy()
        w_backward = w.copy()
        
        # Perturb only the i-th dimension
        w_forward[i] += h
        w_backward[i] -= h
        
        # Compute the central finite difference for the i-th partial derivative
        dw_func[i] = (func(w_forward) - func(w_backward)) / (2 * h)
        
    return dw_func

You can test the function in the cell below:

In [10]:
def run_tests():
    
    # func(w) = sum(w .^ 2)
    func = lambda w: np.sum(w ** 2)

    w_one = np.ones(2)

    dw_func = finite_diff_grad(func, w_one)
    assert len(dw_func) == len(w_one)
    assert np.allclose(dw_func, [2.0, 2.0]) # np.allclose replaces ≈

    # compute gradient of simple uni-variate function:
    dw_uni = finite_diff_grad(lambda w: w[0]**2, np.array([1.0]))
    assert np.allclose(dw_uni, [2.0])

    ### BEGIN TESTS
    # Did students follow the instructions and allocate right arrays without screwing the types?
    w_rand3 = np.random.rand(3)
    dw_func = finite_diff_grad(func, w_rand3)
    assert len(dw_func) == len(w_rand3)
    assert np.allclose(dw_func, 2 * w_rand3)

    # How good are the gradients?
    func_lin = lambda w: np.sum(w)  # finite differences are exact for linear functions
    w_rand5 = np.random.rand(5)
    dw_func = finite_diff_grad(func_lin, w_rand5)
    assert len(dw_func) == len(w_rand5)
    assert np.allclose(dw_func, 1.0) # all(dw_func .≈ 1)

    dw_func = finite_diff_grad(func_lin, w_rand5, h=1.0)
    assert len(dw_func) == len(w_rand5)
    assert np.allclose(dw_func, 1.0)

    func_symmetric = lambda w: np.sum(np.cos(w)) # central difference vanishes around symmetries
    dw_func = finite_diff_grad(func_symmetric, np.zeros(3))
    assert np.sum(dw_func ** 2) <= 1e-6
    ### END TESTS
    
    print("All tests passed successfully!")

# Execute the tests
run_tests()

All tests passed successfully!


### Exercise 1.b)
Investigate the accuracy of `finite_diff_grad` for different grid-sizes `h` on the function
$$ \mathcal L\colon ℝ^q \to ℝ, \; \mathcal L(\mathbf w) = \exp(w_1) + \sum_{i=1}^q w^4_i,$$
where $\mathbf w = [w_1, …, w_q]^T$.

Complete the code in the cell below to test the values $h\in \{10^{-1}, 10^{-2}, …, 10^{-10}\}$.
For each grid-size value $h$, store the error
$$
\left\| Δ_h[\mathcal L](\mathbf w) - \nabla \mathcal L(\mathbf w) \right\|^2
$$
in `fd_grad_errors`. That is, besides the finite-difference approximation you also have to calculate the analytical derivative!

To solve this, we first need to determine the analytical (exact) derivative of the target function by hand so we can compare our approximation against it.

The function is:


$$\mathcal{L}(\mathbf{w}) = \exp(w_1) + \sum_{i=1}^q w_i^4$$

Taking the partial derivative with respect to each element $w_i$, we get:

* For the first element ($w_1$): $\frac{\partial \mathcal{L}}{\partial w_1} = \exp(w_1) + 4w_1^3$
* For all other elements ($w_{i \neq 1}$): $\frac{\partial \mathcal{L}}{\partial w_i} = 4w_i^3$

### What to expect from the results

If we run this code and look at the printed errors, we will likely notice an interesting phenomenon: the error will initially decrease as $h$ gets smaller (going from $10^{-1}$ to around $10^{-5}$).

However, as $h$ gets *too* small (like $10^{-9}$ or $10^{-10}$), the error will suddenly start getting worse again. This is called **floating-point truncation error** (or subtractive cancellation). Because computers can only store so many decimal places, subtracting two nearly identical numbers `func(w + h)` and `func(w - h)` when $h$ is microscopic results in a massive loss of numerical precision. This is exactly why Automatic Differentiation is preferred over Finite Differences for high-precision computing!



In [11]:
import matplotlib.pyplot as plt
def test_func(w):
    # Note: Python uses 0-based indexing, so w_1 is w[0]
    return np.exp(w[0]) + np.sum(w ** 4)

def grad_test_func(w):
    """Return the **true** gradient vector of `test_func` w.r.t. `w`"""
    # Calculate 4 * w^3 for all elements
    grad = 4 * (w ** 3)
    # Add the exp(w_1) term to the first partial derivative
    grad[0] += np.exp(w[0])
    return grad

# grid sizes `h` to test: 10^-1, 10^-2, ..., 10^-10
# np.logspace is perfect for this: start=-1, stop=-10, 10 steps, base 10
fd_grid_sizes = np.logspace(-1, -10, num=10, base=10.0)

# evaluation point, **don't** change
w_fd = np.array([np.pi, 10.0, -np.e / 20.0])

# exact gradient vector `grad_test_func_exact` at `w_fd`
grad_test_func_exact = grad_test_func(w_fd)

# pre-allocate array to store the error values in
fd_grad_errors = np.zeros(len(fd_grid_sizes))

for i, h in enumerate(fd_grid_sizes):
    # compute finite difference gradient
    fd_approx = finite_diff_grad(test_func, w_fd, h)
    
    # store error in `fd_grad_errors[i]`
    # Error is the squared L2 norm: || approx - exact ||^2
    error = np.sum((fd_approx - grad_test_func_exact) ** 2)
    fd_grad_errors[i] = error

# Optional: Print the results to see the U-shaped error curve
print("Grid Size (h) | Squared Error")
print("-" * 30)
for h, err in zip(fd_grid_sizes, fd_grad_errors):
    print(f"{h:1.0e}       | {err:e}")

# 5. Plot the results
plt.figure(figsize=(10, 6))
# Using a log-log scale because both grid sizes and errors span many orders of magnitude
plt.loglog(fd_grid_sizes, fd_grad_errors, marker='o', linestyle='-', color='#1f77b4', linewidth=2, markersize=8)

plt.xlabel('Grid Size ($h$)', fontsize=14)
plt.ylabel('Squared Error ($L_2$ norm)', fontsize=14)
plt.title('Accuracy of Central Finite Difference vs Grid Size ($h$)', fontsize=16)

# Invert x-axis so we read from larger h (left) to smaller h (right)
plt.gca().invert_xaxis()
plt.grid(True, which="both", ls="--", alpha=0.5)

plt.tight_layout()
plt.savefig('error_plot.png')
plt.close()

Grid Size (h) | Squared Error
------------------------------
1e-01       | 1.870079e-01
1e-02       | 1.870016e-05
1e-03       | 1.869833e-09
1e-04       | 1.839230e-13
1e-05       | 3.733338e-14
1e-06       | 8.284538e-12
1e-07       | 3.769613e-10
1e-08       | 7.756835e-08
1e-09       | 4.306811e-07
1e-10       | 1.759772e-04


You can have a look at the error values by plotting them, for example with `Makie`.
Usually, we would expect the error to decrease initially with the grid-size.
But at some point, round-off errors will lead to an increase again.
We can see this in this pre-made plot:
![fd error plot](fd.png)

### Exercise 1.c)

#### Intro

The Julia ecosystem provides many tools for [Automatic Differentiation](https://en.wikipedia.org/wiki/Automatic_differentiation).
`Zygote` is used in the popular machine learning library [`Flux.jl`](https://fluxml.ai/Flux.jl/stable/) 
and implements reverse accumulation to compute loss function gradients.
`ForwardDiff` is a forward-mode library and very robust. It works on functions acting on 
`Real` input.
Lastly, `FiniteDiff` does as the name suggests and computes finite difference derivative
approximations.

<div class="alert alert-block alert-info">
Flux and Lux are destined to move to Enzyme at some point in time.
The project <a href="https://github.com/JuliaDiff/AbstractDifferentiation.jl">AbstractDifferentiation</a>
wants to provide a unified API for many differentiation packages, but lacks caching mechanisms
and is thus not endorsed by SciML libraries (yet).
</div>

In [12]:
import jax
import jax.numpy as jnp
import numdifftools as nd
import numpy as np

Use all the above libraries to compute partial first-order derivatives of the scaled
Gaussian RBF 
$$
m(\mathbf z; a, b, \mathbf c) = b \cdot \exp\left( -\frac{\| \mathbf z - \mathbf c \|^2}{a^2} \right).
$$
The function $m$ (like $m$odel) maps $\mathbf z\in ℝ^n$ to a scalar value.
It is **c**entered at $\mathbf c \in ℝ^n$, and has shape parameter $a>0$, and scaling factor $b \in ℝ$.

In [13]:
def rbf(z, a, b, c):
    """
    Computes the scaled Gaussian RBF.
    z, c are vectors. a, b are scalars.
    """
    # jnp behaves exactly like numpy, but is differentiable by JAX
    r_squared = jnp.sum((z - c) ** 2)
    return b * jnp.exp(-r_squared / a**2)

#### Exercise

Use `FD.gradient`, `Zygote.gradient` and `FiniteDiff.finite_difference_derivative` to
calculate the partial derivatives of $m$ with respect to $\mathbf z$,
$\mathbf c$ and $a$ respectively:
* `FD.gradient`: $\partial m / \partial \mathbf z \in \mathbb{R}^2,$
* `Zygote.gradient`: $\partial m / \partial \mathbf c \in \mathbb{R}^2,$
* `FiniteDiff.finite_difference_derivative`: $\partial m / \partial a \in \mathbb{R}.$

To do so, you can use anonymous functions, e.g. `_z -> rbf(_z, c, a)`. 
**Always return a `Vector`!**

In [14]:
# ---------------------------------------------------------
# 1. Forward-mode AD w.r.t `z` (matches ForwardDiff)
# ---------------------------------------------------------
def dz_rbf_ForwardDiff(z, a, b, c):
    # jax.jacfwd executes forward-mode differentiation.
    # argnums=0 tells it to take the derivative w.r.t the 0th argument (z).
    grad_func = jax.jacfwd(rbf, argnums=0)
    
    # Evaluate and ensure it's returned as a standard array
    return np.array(grad_func(z, a, b, c))

In [15]:
# ---------------------------------------------------------
# 2. Reverse-mode AD w.r.t `c` (matches Zygote)
# ---------------------------------------------------------
def dc_rbf_Zygote(z, a, b, c):
    # jax.grad executes reverse-mode differentiation (standard for ML/Zygote).
    # argnums=3 tells it to take the derivative w.r.t the 3rd argument (c).
    grad_func = jax.grad(rbf, argnums=3)
    
    return np.array(grad_func(z, a, b, c))

In [16]:
# ---------------------------------------------------------
# 3. Finite Differences w.r.t `a` (matches FiniteDiff)
# ---------------------------------------------------------
def da_rbf_FiniteDiff(z, a, b, c):
    # Cast `a` to a float, just like the Julia hint suggests
    a = float(a)
    
    # We use a lambda function to freeze z, b, and c, creating a 
    # function of a single variable `_a`. This matches the Julia 
    # anonymous function hint `_z -> rbf(_z, c, a)`.
    rbf_a_only = lambda _a: rbf(z, _a, b, c)
    
    # nd.Derivative computes the finite difference for a scalar input
    df_func = nd.Derivative(rbf_a_only)
    
    # Evaluate the derivative at our specific `a`
    derivative_val = df_func(a)
    
    # The instructions explicitly state: "Always return a Vector!"
    # Since the derivative w.r.t a scalar is a scalar, we wrap it in a 1D array.
    return np.array([derivative_val])

You should now be able to evaluate the partial gradients at 
$(\mathbf z, \mathbf c, a) = (\mathbf 1, \mathbf 0, 1)$:

Deriving the exact analytical gradients by hand is the perfect way to prove that our Automatic Differentiation and Finite Difference methods are giving us the correct values.

Let's break down the math using the chain rule. We start with our scaled Gaussian RBF:

$$m(\mathbf{z}; a, b, \mathbf{c}) = b \cdot \exp\left(-\frac{\|\mathbf{z} - \mathbf{c}\|^2}{a^2}\right)$$

To make the chain rule easier to read, let's define the squared distance as $R = \|\mathbf{z} - \mathbf{c}\|^2 = \sum_{i=1}^n (z_i - c_i)^2$. So our function is $m = b \cdot \exp\left(-\frac{R}{a^2}\right)$.

---

### 1. Partial Derivative with respect to $\mathbf{z}$

We apply the chain rule: the derivative of the exponential function, multiplied by the derivative of the inside with respect to $\mathbf{z}$.

* $\frac{\partial m}{\partial \mathbf{z}} = b \cdot \exp\left(-\frac{R}{a^2}\right) \cdot \left( -\frac{1}{a^2} \right) \cdot \frac{\partial R}{\partial \mathbf{z}}$
* The derivative of the squared distance $R$ with respect to $\mathbf{z}$ is $2(\mathbf{z} - \mathbf{c})$.

Combining them:


$$\frac{\partial m}{\partial \mathbf{z}} = -\frac{2b}{a^2} (\mathbf{z} - \mathbf{c}) \exp\left(-\frac{\|\mathbf{z} - \mathbf{c}\|^2}{a^2}\right)$$

> **Verifying the Code Test:** > The hidden test for `ForwardDiff` evaluated $\mathbf{z}=[1, 2]$, $\mathbf{c}=[5, 6]$, $a=3$, and $b=4$.
> * $R = (1-5)^2 + (2-6)^2 = (-4)^2 + (-4)^2 = 32$
> * $\frac{\partial m}{\partial \mathbf{z}} = -\frac{2(4)}{3^2} \cdot [-4, -4] \cdot \exp\left(-\frac{32}{9}\right) = \left[ \frac{32}{9} e^{-32/9}, \frac{32}{9} e^{-32/9} \right]$
> * $\frac{32}{9} e^{-32/9} \approx 0.1014$
> * *This perfectly matches the test assertion `isapprox(g, 0.1; rtol=1e-1)`.*
> 
> 

### 2. Partial Derivative with respect to $\mathbf{c}$

This is almost identical to the derivative for $\mathbf{z}$. The only difference is the inner derivative of the squared distance $R$ with respect to $\mathbf{c}$, which yields a negative sign: $-2(\mathbf{z} - \mathbf{c})$.

$$\frac{\partial m}{\partial \mathbf{c}} = \frac{2b}{a^2} (\mathbf{z} - \mathbf{c}) \exp\left(-\frac{\|\mathbf{z} - \mathbf{c}\|^2}{a^2}\right) = - \frac{\partial m}{\partial \mathbf{z}}$$

> **Verifying the Code Test:**
> Since this is just the exact negative of the $\mathbf{z}$ gradient, evaluating it at the same test points ($\mathbf{z}=[1, 2]$, $\mathbf{c}=[5, 6]$, $a=3$, $b=4$) gives us $-0.1014$.
> * *This perfectly matches the `Zygote` test assertion `isapprox(g, -0.1; rtol=1e-1)`.*
> 
> 

### 3. Partial Derivative with respect to $a$

Here, we are taking the derivative with respect to the denominator of the exponent.

* $\frac{\partial m}{\partial a} = b \cdot \exp\left(-\frac{R}{a^2}\right) \cdot \frac{\partial}{\partial a}\left(-R \cdot a^{-2}\right)$
* Using the power rule: $\frac{\partial}{\partial a}\left(-R \cdot a^{-2}\right) = -R \cdot (-2a^{-3}) = \frac{2R}{a^3}$

Combining them:


$$\frac{\partial m}{\partial a} = b \cdot \frac{2\|\mathbf{z} - \mathbf{c}\|^2}{a^3} \exp\left(-\frac{\|\mathbf{z} - \mathbf{c}\|^2}{a^2}\right)$$

> **Verifying the Code Test:**
> The `FiniteDiff` block tested the variables $\mathbf{z}=[1, 1]$, $\mathbf{c}=[0, 0]$, $a=1$, and $b=1$.
> * $R = (1-0)^2 + (1-0)^2 = 2$
> * $\frac{\partial m}{\partial a} = 1 \cdot \frac{2(2)}{1^3} \cdot \exp\left(-\frac{2}{1^2}\right) = 4e^{-2}$
> * $4e^{-2} \approx 0.5413411329...$
> * *This perfectly matches the 14-decimal precision assertion `0.5413411329152712`!*
> 
> 

---

It is always deeply satisfying when the manual calculus, the exact Automatic Differentiation graph, and the numerical approximations all converge on the exact same numbers.

In [ ]:
def run_gradient_tests():
    # Setup initial variables to match `ones(2)`, `zeros(2)`, etc.
    z = jnp.ones(2)
    c = jnp.zeros(2)
    a = 1.0
    b = 1.0

    # ---------------------------------------------------------
    # Test 1: Forward-mode AD w.r.t `z`
    # ---------------------------------------------------------
    grad_z = dz_rbf_ForwardDiff(z, a, b, c) # Call the function to compute the gradient w.r.t z
    
    # `@assert ... isa Vector` -> Check if it's a 1D numpy array
    assert isinstance(grad_z, np.ndarray) and grad_z.ndim == 1
    assert len(grad_z) == 2
    
    # `isapprox(g, 0.1; rtol=1e-1)`
    test_z = jnp.array([1.0, 2.0])
    test_c = jnp.array([5.0, 6.0])
    grad_z_test = dz_rbf_ForwardDiff(test_z, 3.0, 4.0, test_c)
    assert np.allclose(grad_z_test, 0.1, rtol=1e-1)


    # ---------------------------------------------------------
    # Test 2: Reverse-mode AD w.r.t `c`
    # ---------------------------------------------------------
    grad_c = dc_rbf_Zygote(z, a, b, c)
    
    assert isinstance(grad_c, np.ndarray) and grad_c.ndim == 1
    assert len(grad_c) == 2
    
    # `isapprox(g, -0.1; rtol=1e-1)`
    grad_c_test = dc_rbf_Zygote(test_z, 3.0, 4.0, test_c)
    assert np.allclose(grad_c_test, -0.1, rtol=1e-1)


    # ---------------------------------------------------------
    # Test 3: Finite Differences w.r.t `a`
    # ---------------------------------------------------------
    grad_a = da_rbf_FiniteDiff(z, a, b, c)
    
    assert isinstance(grad_a, np.ndarray) and grad_a.ndim == 1
    assert len(grad_a) == 1
    
    # `only(...) ≈ 0.5413411329152712`
    # only() in Julia extracts the single element from a collection.
    # In Python, we just index the first element [0].
    assert np.isclose(grad_a[0], 0.5413411329152712)
    
    print("All tests passed successfully!")

# Execute the tests
run_gradient_tests()

All tests passed successfully!


### Exercise 1d)

#### Intro

Just like on the last exercise sheet, we now compose a complete RBF approximation model 
$h \colon ℝ^n \to ℝ$
as the sum of $N_\text{RBF} \in ℕ$ RBFs with different parameters
$$
h(\mathbf z) = 
h(\mathbf z; \mathbf w)
= \sum_{i=1}^{N_{\text{RBF}}} m(\mathbf z; a_i, b_i, \mathbf c_i).
$$
Here, $\mathbf w$ is the complete (flattened) parameter vector of $h$ holding 
$a_i, b_i$ and $\mathbf c_i$ for $i=1,…, N_{\text{RBF}}$.

In [18]:
def h_rbf(z, w):
    """
    Given a flattened parameter vector `w` (of suitable size), return the value
    of the complete RBF model.
    """
    dim_z = len(z)
    assert dim_z > 0, "Input vector z must not be empty"

    # 1 center (size dim_z) + shape param (1) + coefficient (1) per RBF kernel
    num_rbf_params = dim_z + 2 

    # check length of `w` vector
    len_w = len(w)
    assert len_w >= num_rbf_params, "Parameter vector w is too short"
    assert len_w % num_rbf_params == 0, "Length of w must be a multiple of num_rbf_params"
    
    # integer division in Python is //
    num_rbfs = len_w // num_rbf_params

    # unflatten and sum RBF kernels
    h_val = 0.0
    
    # Pythonic loop: step by 'num_rbf_params' through the array
    # This replaces the manual `j += num_rbf_params` from the Julia code
    # eg [a1,b1,c1,a2,b2,c2, ..]
    for j in range(0, len_w, num_rbf_params):
        a = w[j]
        b = w[j+1]
        
        # Python slicing is end-exclusive: [start : start + length]
        c = w[j+2 : j+2+dim_z]
        
        h_val += rbf(z, a, b, c)

    return h_val


<div class="alert alert-block alert-info">
Typically, models provided by some machine learning (like `Flux` or `Lux`), would not store 
their parameters in a flattened vector.
Not only is the model implementation more intuitive that way, it also allows for optimized
evaluation on large data sets.
For example, Flux models return `Params` objects, that usually store arrays of varying dimensions, 
dependent on the model structure.
The cool thing about `Zygote` is, that it keeps that structure when taking gradients.
So the gradient of some loss function with respect to parameters that are stored in a matrix 
is a matrix, which enables convenient updating.
<br/>
Both libraries offer tools for flattening to use external optimizers.
</div>

Now assume that $\mathbf Z \in ℝ^{n \times N}$ is a matrix, 
the columns of which hold feature vectors for labeled data.
The labels are in $\mathbf y\in ℝ^{N}$.
We want to use $h(•; \mathbf w)$ to approximate the data and the arrays induce a loss function 
$$
\mathcal L(\mathbf w) = \mathcal L(\mathbf w; \mathbf Z, \mathbf y) = \frac{1}{N}
\sum_{j=1}^{N}
(\mathbf y_j - h(\mathbf z_j, \mathbf w)) ^2
$$


Here is how to compute that value:

In [19]:
def rbf_mse(w, Z, y):
    """
    Computes the Mean Squared Error of the RBF model.
    w: Flattened parameter vector.
    Z: Matrix where each column is a feature vector.
    y: Array of true labels/target values.
    """
    mse_val = 0.0
    mse_divisor = 0
    
    # Z.T transposes the matrix so we can iterate over its columns.
    # zip() pairs each column `zj` with its corresponding label `yj`.
    for zj, yj in zip(Z.T, y):
        # 1. Make a prediction using our RBF model
        prediction = h_rbf(zj, w)
        
        # 2. Calculate the squared error for this specific point
        error_squared = (prediction - yj) ** 2
        
        # 3. Add to our running total
        mse_val += error_squared
        mse_divisor += 1
        
    # 4. Divide by total number of points to get the "Mean"
    mse_val /= mse_divisor
    
    return mse_val

#### Exercise

Now it's your turn.

Complete the cell below and use `Zygote.pullback` to compute the mean squared error **and the loss gradient**
with respect to $\mathbf w$ at the same time.
The syntax is 
```julia
result, back = Zygote.pullback( some_func, func_args )
```
and `back` is the **pullback function** that gives a tuple of gradient objects when called 
with seed `one(result)` (see lecture video on reverse-mode AD).

Oftentimes, `some_func` is actually anonymous, e.g., to compute partial gradients only.
In that case, it *can* be more convenient to use a `do`-block:
```julia
result, back = Zygote.pullback( func_arg_val1, func_arg_val2, … ) do func_arg_name1, func_arg_name2, …
    # function body acting on `func_arg_name1` etc.
    local_result
end
```
---

*(Note: For JAX to automatically differentiate `rbf_mse`, the operations inside it must be built using `jax.numpy` rather than standard Python loops and `math` libraries, as JAX needs to trace the mathematical operations!)*

### Explaining the Concept: Why compute both at the same time?

When training a model, you need two things:

1. **The Loss:** To track how well the model is doing.
2. **The Gradient:** To know how to update the weights (`w`) to make the model better.

You *could* calculate them completely separately:

```python
loss = rbf_mse(w, Z, y)
grad = jax.grad(rbf_mse)(w, Z, y)

```

**However, this is incredibly inefficient.** Calculating a gradient using the chain rule requires all the intermediate values that were generated during the forward pass (when the loss was calculated). If you run them separately, your computer evaluates the entire function to get the loss, throws all the intermediate math away, and then has to *recalculate* all of those intermediate steps from scratch just to figure out the gradient.

#### The "Pullback" (Reverse-Mode AD)

The solution is the **pullback** (also called a Vector-Jacobian Product, or the backward pass). Here is how it works under the hood:

1. **The Forward Pass:** The computer runs your `rbf_mse` function. But as it calculates the final number, it secretly builds a "computational graph" in memory, saving every intermediate variable and mathematical operation it performed.
2. **The Return:** It returns the final scalar `result` (the loss), alongside a `back` function (the pullback) which contains that saved graph.
3. **The Seed:** You provide a "seed" to the pullback. For a scalar loss function, this seed is always $1$ (because the derivative of the loss with respect to itself is $\frac{\partial \mathcal{L}}{\partial \mathcal{L}} = 1$).
4. **The Backward Pass:** The pullback function takes that $1$ and pushes it *backwards* through the saved computational graph, applying the chain rule step-by-step from the output all the way back to your input weights $\mathbf{w}$.

By using pullbacks/`value_and_grad`, you reuse the exact same computational graph for both the answer and the derivative, saving massive amounts of compute time.

Here is the exercise:

In [21]:
def rbf_mse_and_grad(w, Z, y):
    # jax.value_and_grad creates a function that returns (value, gradient)
    # argnums=0 means we only want the gradient w.r.t the first argument (w)
    loss_val_and_grad_fn = jax.value_and_grad(rbf_mse, argnums=0)
    
    # Evaluate it
    loss_val, grad_w = loss_val_and_grad_fn(w, Z, y)
    
    return loss_val, grad_w

In [20]:
#method 2 to compute the gradient using jax.grad separately
def rbf_mse_and_grad_pullback(w, Z, y):
    # jax.vjp executes the forward pass and returns a pullback function
    loss_val, pullback_fn = jax.vjp(rbf_mse, w, Z, y)
    
    # We call the pullback function with our "seed" (1.0 for a scalar)
    # It returns a tuple of gradients for ALL inputs (w, Z, y).
    # We only care about the gradient for `w`, which is at index 0.
    seed = 1.0 
    gradients = pullback_fn(seed)
    grad_w = gradients[0]
    
    return loss_val, grad_w

Let's test the implementation:

### Explaining the Concepts

This test block verifies two critical properties of our gradient implementation.

#### Concept 1: Forward vs. Reverse Mode Agreement

Lines 23-25 are the most important check. We calculated `dL` using Reverse-Mode AD (the pullback/Zygote approach), which is highly efficient. But just to be absolutely sure it is mathematically correct, we also calculate `_dL` using Forward-Mode AD (`FD.gradient` or `jax.jacfwd`).

* **Forward Mode** tracks how one specific input changes every possible output. It is slow when you have thousands of parameters (`w`) and only one output (loss), but it is very robust.
* **Reverse Mode** tracks how one specific output is affected by every possible input. It is the standard for machine learning because it gives you all the parameter gradients in a single backward pass.

If both completely different algorithms yield the exact same gradient vector, we know our implementation is flawless.

#### Concept 2: The "Flat" Gradient Check

The final "Hidden Tests" block sets up a very specific scenario:

1. All input data (`Z`) is zero.
2. All target labels (`y`) are zero.
3. For the RBF parameters (`w`), it sets all the "width" parameters ($a$) to 1, and everything else (centers $c$, heights $b$) to zero.

Because the heights ($b$) are 0, every RBF curve outputs exactly 0.
Because the targets ($y$) are exactly 0, the prediction is perfect.
Because the prediction is perfect, the Loss is exactly $0$.

If the loss is at its absolute minimum possible value ($0$), you are standing at the very bottom of the "valley" of the loss landscape. If you take a derivative at the bottom of a valley, the slope should be perfectly flat. Therefore, the test asserts that every single value in the gradient vector (`dL`) is effectively zero (`< 1e-5`).

In [22]:
def run_final_tests():
    # ---------------------------------------------------------
    # PART 1: Random Data Initialization
    # ---------------------------------------------------------
    # JAX handles random numbers differently than NumPy/Julia. 
    # It requires explicitly passing a 'key' state around.
    key = jax.random.PRNGKey(31415)
    
    # We split the key to generate independent random variables
    key_Z, key_y, key_w = jax.random.split(key, 3)
    
    n = 3 # feature dimension
    N = 4 # number of samples
    
    # Random training data
    Z = jax.random.uniform(key_Z, shape=(n, N))
    y = jax.random.uniform(key_y, shape=(N,))
    
    # Build flattened parameter vector
    num_rbfs = 10
    w = jax.random.uniform(key_w, shape=((n + 2) * num_rbfs,))
    
    # ---------------------------------------------------------
    # PART 2: Testing the Implementation
    # ---------------------------------------------------------
    
    # Compute loss on random data normally
    _L = rbf_mse(w, Z, y)
    
    # Compute loss and loss gradient using our combined function
    L, dL = rbf_mse_and_grad(w, Z, y)
    
    # Assertions
    # JAX arrays are considered numbers for this purpose
    assert isinstance(L, (float, jax.Array, np.ndarray)) and L.ndim == 0
    assert jnp.isclose(L, _L)
    assert isinstance(dL, (jax.Array, np.ndarray)) and dL.ndim == 1
    
    # Check consistency with Forward-mode AD
    # We use jax.jacfwd to compute the exact gradient w.r.t 'w' 
    # to mimic Julia's FD.gradient
    forward_grad_fn = jax.jacfwd(rbf_mse, argnums=0)
    _dL = forward_grad_fn(w, Z, y)
    
    assert jnp.allclose(_dL, dL)
    
    # ---------------------------------------------------------
    # PART 3: Hidden Tests (Zero Data)
    # ---------------------------------------------------------
    dim_data = 6
    num_data = 6
    
    Z_zero = jnp.zeros((dim_data, num_data))
    y_zero = jnp.zeros(num_data)
    
    n_params = (dim_data + 2) * num_rbfs
    w_test = np.zeros(n_params) # Use NumPy initially for easier mutation
    
    # Julia is 1-indexed, Python is 0-indexed.
    # The Julia code sets w[i] = 1 when i % (dim_data + 2) == 1.
    # In 0-indexed logic, this corresponds to the very first element 
    # of every block (index 0, 8, 16...), which is i % (dim_data + 2) == 0
    for i in range(len(w_test)):
        if i % (dim_data + 2) == 0:
            w_test[i] = 1.0
            
    # Convert back to JAX array for evaluation
    w_test = jnp.array(w_test)
            
    _L_test = rbf_mse(w_test, Z_zero, y_zero)
    L_test, dL_test = rbf_mse_and_grad(w_test, Z_zero, y_zero)
    
    assert jnp.isclose(_L_test, L_test)
    assert jnp.all(jnp.abs(dL_test) < 1e-5)
    
    print("All final tests passed successfully!")

# Execute the tests
run_final_tests()

All final tests passed successfully!


## Exercise 2 - Momentum Descent

By extending the classical (stochastic) gradient descent update rule by a momentum term,
we can sometimes observe an improved convergence rate.
Polyak's Heavy Ball momentum is one of the best-known examples of this class of algorithms.
The parameter update for a loss 
$\mathcal L\colon ℝ^q \to ℝ, \mathbf w \mapsto \mathcal L(\mathbf w)$ 
in iteration $k\in ℕ_0$ is 
$$
\mathbf w^{(k+1)}
\leftarrow
\mathbf w^{(k)}
-
α \nabla \mathcal L(\mathbf w^{(k)})
+ 
κ
(\mathbf w^{(k)} - \mathbf w^{(k-1)}).
$$
We assume to start with $\mathbf w^{(0)}$.
The algorithm is instantiated with $\mathbf w^{(-1)} = \mathbf w^{(0)}$, i.e., without momentum in the first iteration.

### The Concept: Momentum Descent (Polyak's Heavy Ball)

Standard Gradient Descent updates the current position based *only* on the slope (gradient) at the exact spot you are currently standing. While simple, this can be painfully slow, especially when navigating functions shaped like narrow valleys or ravines (like the Rosenbrock function). In a narrow valley, standard gradient descent tends to zig-zag violently across the steep walls while making very little forward progress along the flat valley floor.

**Momentum** solves this by adding "physical inertia" to the optimization process. Imagine rolling a heavy iron ball down a hill. It doesn't just instantly change direction based on the slope underneath it; it builds up speed in the direction it has been traveling.

Let's break down the mathematical update rule:


$$\mathbf{w}^{(k+1)} \leftarrow \mathbf{w}^{(k)} - \alpha \nabla \mathcal{L}(\mathbf{w}^{(k)}) + \kappa(\mathbf{w}^{(k)} - \mathbf{w}^{(k-1)})$$

Here is what each piece represents:

* **$\mathbf{w}^{(k)}$**: Your current position.
* **$- \alpha \nabla \mathcal{L}(\mathbf{w}^{(k)})$**: The standard gradient descent step. It says, "Take a step of size $\alpha$ down the steepest part of the current slope."
* **$+ \kappa(\mathbf{w}^{(k)} - \mathbf{w}^{(k-1)})$**: The **momentum term**. The vector $(\mathbf{w}^{(k)} - \mathbf{w}^{(k-1)})$ represents the exact direction and distance of the *previous* step you took. By multiplying this by a momentum coefficient $\kappa$ (usually between 0.5 and 0.99) and adding it to your update, you force the algorithm to keep moving in the direction it was already going.

**The Result:** The oscillations across the steep walls cancel themselves out (because they alternate directions), while the movement along the valley floor accumulates and accelerates, drastically speeding up convergence.

For the very first step, there is no "previous step" to build momentum from. This is why the algorithm initializes with $\mathbf{w}^{(-1)} = \mathbf{w}^{(0)}$, meaning the momentum term calculates to $0$ on iteration $k=0$.

To test our algorithm, we are again considering the _Rosenbrock function_:
$$
\mathcal L(\mathbf w) = (a - w_1)^2 + b(w_2 - w_1^2)^2.
$$

In [2]:
def rosenbrock_2D(w, a=1, b=100):
    # w is expected to be a list, tuple, or numpy array with at least 2 elements
    # Python uses 0-based indexing: w[0] is w_1, w[1] is w_2
    return (a - w[0])**2 + b * (w[1] - w[0]**2)**2

### Exercise 2a)
Implement the **exact** gradient of the Rosenbrock function. You can compute it by hand or use `ForwardDiff` or `Zygote`.

In [3]:
import jax
import jax.numpy as jnp

def dw_rosenbrock_2D(w, a=1.0, b=100.0):
    ### BEGIN SOLUTION
    
    # jax.grad returns a NEW function that calculates the gradient 
    # of 'rosenbrock_2D' with respect to its first argument (argnums=0, which is 'w').
    grad_function = jax.grad(rosenbrock_2D, argnums=0)
    
    # Execute the generated gradient function
    gradient = grad_function(w, a, b)
    
    ### END SOLUTION
    return gradient

# --- Example Usage ---
# Note: JAX requires inputs to be float types (e.g., 1.0 instead of 1)
w_test = jnp.array([1.5, 2.0])
print(f"Exact Gradient via AD: {dw_rosenbrock_2D(w_test)}")

Exact Gradient via AD: [151. -50.]


The global optimum is known to be $\mathbf x = [a, a^2]^\top$ and we can check for consistency,
if the value is zero and the gradient vanishes:

In [5]:
import numpy as np

# Wrap in a function to mimic Julia's 'let' local scope block
def run_consistency_tests():
    # 1. Test standard global minimum at [1, 1]
    L = rosenbrock_2D(np.array([1.0, 1.0]))
    assert np.isclose(L, 0.0), "Loss should be 0 at the global optimum"
    
    dL = dw_rosenbrock_2D(np.array([1.0, 1.0]))
    assert np.allclose(dL, 0.0), "Gradient should vanish (be 0) at the global optimum"
    
    ### BEGIN TESTS
    
    # 2. Test optimum at [a, a^2] for a random parameter 'a'
    a = np.random.rand()
    w_optimum = np.array([a, a**2])
    
    L = rosenbrock_2D(w_optimum, a=a)
    assert np.isclose(L, 0.0)
    
    dL = dw_rosenbrock_2D(w_optimum, a=a)
    assert np.allclose(dL, 0.0, atol=1e-5) # atol added for floating point safety
    
    # 3. Test gradient consistency on random inputs
    w = np.random.rand(2)
    a = np.random.rand()
    b = np.random.rand()
    
    # Your implemented gradient
    dL = dw_rosenbrock_2D(w, a=a, b=b)
    
    # The Automatic Differentiation gradient (acting as ForwardDiff)
    # jax.grad computes the exact gradient with respect to the first argument (w)
    FD_gradient = jax.grad(rosenbrock_2D, argnums=0)
    dL_AD = FD_gradient(w, a, b)
    
    # Assert they are approximately equal (≈)
    assert np.allclose(dL, dL_AD), "Implemented gradient does not match AD gradient!"
    
    ### END TESTS
    
    print("All tests passed successfully!")

# Execute the local scope block
run_consistency_tests()

All tests passed successfully!


### Exercise 2b)

We now want to implement a training algorithm `momentum_descent` with fixed
meta-parameters.
Besides the meta parameters, the function is provided with functions
`loss`, `dw_loss` and initial parameters `w`.

Assume `loss` to return a scalar loss value when called as `loss(w)`, and `dw_loss`
to return a loss gradient vector.

### The Concept: Implementing Polyak's Heavy Ball

To implement the Heavy Ball momentum algorithm, we have to keep track of our "history"—specifically, where we were one step ago.

Here is how the algorithm breaks down into code:

1. **Initialization (`w_prev` and `momentum`):** The formula requires the previous position $\mathbf{w}^{(k-1)}$. On the very first iteration ($k=0$), there is no previous position. The standard mathematical trick is to assume the algorithm was standing completely still before it started. Therefore, we set $\mathbf{w}^{(-1)} = \mathbf{w}^{(0)}$. Consequently, the initial momentum (the difference between the two) is exactly $0$.
2. **The Update Loop:** Inside the loop, we do three things in a very specific order:
* Calculate the current gradient $\nabla \mathcal{L}(\mathbf{w}^{(k)})$.
* Calculate the momentum term by finding the difference between our current position and our previous position ($\mathbf{w}^{(k)} - \mathbf{w}^{(k-1)}$).
* Store our *current* position in `w_prev` so it is ready for the next loop.
* Finally, update `wk` by applying the standard gradient descent step **minus** the gradient, **plus** the scaled momentum.

Fill in the cell below according to the comments to finalize the algorithm implementation:

In [6]:
def momentum_descent(loss, dw_loss, w, num_iter=10, alpha=0.1, kappa=0.1):
    # wk = copy(w) # do not modify the initial vector!
    # We use numpy to ensure it's a float array and copy it.
    wk = np.array(w, dtype=float).copy() 
    
    ## It is good practice to preallocate memory before any loop.
    ## Hence, initialize an object `w_prev` for the previous parameters and set
    ## the values according to the formula for the Heavy Ball scheme.
    ## Additionally, allocate a vector `momentum` for the momentum term,
    ## i.e., the difference between two consecutive weight vectors.
    
    ### BEGIN SOLUTION
    # According to the scheme, w^{(-1)} = w^{(0)}
    w_prev = wk.copy()
    
    # Initialize momentum as a vector of zeros with the same shape as wk
    momentum = np.zeros_like(wk)
    ### END SOLUTION
    
    ## Finally, do the iterations by completing the code in the for loop below.
    for k in range(num_iter):
        ## 1) Obtain a loss gradient for the current `wk`.
        ## 2) Modify `momentum`, `w_prev` and `wk` according to the Heavy Ball formulas.
        
        ### BEGIN SOLUTION
        # 1) Get the gradient at the current position
        grad = dw_loss(wk)
        
        # 2) Calculate the momentum term (current position - previous position)
        momentum = wk - w_prev
        
        # Save the current position to w_prev BEFORE we update it
        w_prev = wk.copy()
        
        # Apply the Heavy Ball update rule:
        # w_next = w_current - alpha * gradient + kappa * momentum
        wk = wk - alpha * grad + kappa * momentum
        ### END SOLUTION
        
    ## return last parameters
    return wk

### Exercise 2c)

Minimize the Rosenbrock function with $a=1$ and $b=100$ using your momentum algorithm. Use the same method to compare the results to those of the standard gradient descent (think on how to choose the hyper-parameters to achieve this!)

Perform 100 iterations, starting at `w0_rb`:

In [7]:
import numpy as np

num_iter_rb = 100

## some initial guess:
# Python's numpy provides the value of pi
w0_rb = np.array([-np.pi / 2, np.pi / 4]) # don't change!

## obtain loss function from `rosenbrock_2D` for `a=1` and `b=100` and assign it to `loss_rb`.
### BEGIN SOLUTION
# Create a lambda function that fixes a=1 and b=100
loss_rb = lambda w: rosenbrock_2D(w, a=1.0, b=100.0)
### END SOLUTION


## likewise, obtain loss gradient function `dw_loss_rb` from `rosenbrock_2D` for `a=1` and `b=100`
### BEGIN SOLUTION
dw_loss_rb = lambda w: dw_rosenbrock_2D(w, a=1.0, b=100.0)
### END SOLUTION


## here you can see how we do 100 iterations of momentum descent:
alpha_momentum = 1e-3
kappa_momentum = 0.7

wopt_rb_momentum = momentum_descent(
    loss_rb, dw_loss_rb, w0_rb,
    num_iter=num_iter_rb, alpha=alpha_momentum, kappa=kappa_momentum
)


## exercise: compare against steepest descent!
alpha_sd = alpha_momentum

### BEGIN SOLUTION
# Standard gradient descent has zero momentum
kappa_sd = 0.0
### END SOLUTION

wopt_rb_sd = momentum_descent(
    loss_rb, dw_loss_rb, w0_rb,
    num_iter=num_iter_rb, alpha=alpha_sd, kappa=kappa_sd
)

# --- Optional: Print the results to see the difference ---
print(f"Standard GD Final Loss: {loss_rb(wopt_rb_sd):.4f}")
print(f"Momentum Final Loss:    {loss_rb(wopt_rb_momentum):.4f}")

Standard GD Final Loss: 3.2421
Momentum Final Loss:    0.1047


If your implementation works as it should, the solution trajectories look something like this:
![momentum descent with rosenbrock](momentum_rb.png)

### Exercise 2d)

Finally, in the cell below, test several configurations of the descent algorithm for 
optimization of the Rosenbrock function $\mathcal L$ with $a=1$ and $b=100$.

For `w0_rb` from above, perform 50 iterations of momentum descent for the 
meta-parameters $(α, κ)$ in `alpha_kappa_configs`.
Among those tuples, determine the tuple `(alpha_best, kappa_best)` that achieves the smallest
function value after 50 iterations.


In [9]:
import itertools
import numbers # For testing the type

## these are the meta-parameters to investigate
alphas = (1e-4, 1e-5, 1e-6, 1e-7)
kappas = (0.6, 0.2, 0.1, 1e-3, 1e-4, 1e-6)

# itertools.product creates pairs of every possible combination
alpha_kappa_configs = list(itertools.product(alphas, kappas))

## at the end of this cell block, you should have assigned fitting values to this tuple:
(alpha_best, kappa_best) = (-1.0, -1.0)

### BEGIN SOLUTION
# Initialize the best loss with infinity so any real loss will be smaller
best_loss = float('inf')

# Iterate through every configuration pair
for current_alpha, current_kappa in alpha_kappa_configs:
    
    # Run the momentum descent for 50 iterations
    # (Assuming loss_rb, dw_loss_rb, and w0_rb are still defined from the previous cell)
    w_final = momentum_descent(
        loss_rb, dw_loss_rb, w0_rb,
        num_iter=50, alpha=current_alpha, kappa=current_kappa
    )
    
    # Evaluate how well this configuration did
    current_loss = loss_rb(w_final)
    
    # If it's the lowest loss we've seen so far, record it
    if current_loss < best_loss:
        best_loss = current_loss
        alpha_best = current_alpha
        kappa_best = current_kappa

### END SOLUTION

# --- Testing Block ---
# In Python, we can check if a variable is a number using isinstance and numbers.Real
assert isinstance(alpha_best, numbers.Real)
assert isinstance(kappa_best, numbers.Real)

### BEGIN TESTS
assert (alpha_best, kappa_best) == (1e-4, 0.6), "Incorrect optimal parameters found!"
print(f"All tests passed! Optimal Configuration: alpha={alpha_best}, kappa={kappa_best}")
### END TESTS

All tests passed! Optimal Configuration: alpha=0.0001, kappa=0.6


In [10]:
assert isinstance(alpha_best, numbers.Real)
assert isinstance(kappa_best, numbers.Real)

### BEGIN TESTS
assert (alpha_best, kappa_best) == (1e-4, 0.6)
### END TESTS

## Exercise 3 - Newton's Method

Newton's method is a root finding algorithm.
Consider the function $\mathbf m\colon ℝ^q \to ℝ^q$.
Then Newton's method tries to find $\mathbf w^* \in ℝ^q$ with $\mathbf m(\mathbf w^*) = \mathbf 0$.
The update rule is 
$$
\mathbf w_{k+1} = \mathbf w_k - \left(\nabla \mathbf m(\mathbf w_k)\right)^{-1} \mathbf m(\mathbf w_k),
\quad 
k\in \mathbb N_0,
$$
where $\nabla \mathbf m(\mathbf w_k)$ is the Jacobian of $\mathbf m$ at $\mathbf w_k$.

Instead of computing the inverse in the right-most term, rather substitute it by $\mathbf v$ and solve a linear equation system
$$
\nabla \mathbf m(\mathbf w_k) \mathbf v = - \mathbf m(\mathbf w_k)
$$
to obtain $\mathbf w_{k+1} = \mathbf v + \mathbf w_k$.

### Exercise 3a)

In the cell below, use the formulas from above to implement Newton's root finding algorithm:

In [ ]:
function newton_opt(
    func,       # the function ``m`` from above
    jac_func,   # a function to obtain the jacobian of ``m``
    w0          # initial guess for the root
    ;
    num_iter=10,    # number of iterations
    abs_tol=0,      # absolute stopping tolerance
)
    ## initialize `wk`
    wk = copy(w0)
    
    for k=1:num_iter
        ## evaluate `func` at `wk` and assign results to `mk`
        ### BEGIN SOLUTION
        
        ### END SOLUTION
        
        ## if norm squared of `mk` is <= `abs_tol`, then break:
        ### BEGIN SOLUTION
      
        ### END SOLUTION

        ## compute jacobian at `wk` and store result in `d_mk`
        ## then set `v` to solve `d_mk * v = -mk`
        ### BEGIN SOLUTION
       
        ### END SOLUTION

        ## finally, update `wk` with `v` to store values for iteration `k+1`
        ### BEGIN SOLUTION
       
        ### END SOLUTION
    end

    return wk
end

Let us test your algorithm with a simple example. 
The Jacobian of $\mathbf m( w_1, w_2 ) = [w_1^2, w_2^2]$ is 
$$
\begin{bmatrix}
    2w_1 & 0 \\
    0 & 2w_2
\end{bmatrix}.
$$
Newton's algorithm should approximate $\mathbf w^* = [0, 0]^T$:

In [ ]:
let
    ## test function:
    m = w -> w .^ 2
    ## jacobian:
    dw_m = w -> [
        2*w[1] 0;
        0 2*w[2]
    ]
    
    ## inital guess
    w0 = [1.0, -3.0]
    ## call newton root finder:
    wopt = newton_opt(m, dw_m, w0; num_iter=20, abs_tol=0)
    @assert sum(m(wopt) .^ 2) <= 1e-10

    ## test stopping criterion:
    _wopt = newton_opt(m, dw_m, w0; num_iter=20, abs_tol=1e-3)
    @assert sum(m(_wopt) .^ 2) <= 1e-3
    @assert m(wopt) <= m(wopt)
end

### Exercise 3b)

We want to use Newton's method to **minimize** the Rosenbrock function.
As an optimization algorithm for $\mathcal L\colon ℝ^q \to ℝ$, Newton's root finding scheme is applied to the function 
$$
\mathbf w \mapsto \nabla \mathcal L(\mathbf w)
$$
Hence, we need the Jacobian of the gradient of the Rosenbrock function for `newton_opt`, which is the _Hessian matrix_.

### The Concept: The Hessian Matrix

To use Newton's Method for optimization, we need more than just the slope (gradient); we need to know the *curvature* of the function. This is provided by the **Hessian matrix**, which is the square matrix of second-order partial derivatives.

For a 2D function like the Rosenbrock function $\mathcal{L}(w_1, w_2)$, the Hessian $H$ is a $2 \times 2$ matrix:


$$H = \begin{bmatrix} \frac{\partial^2 \mathcal{L}}{\partial w_1^2} & \frac{\partial^2 \mathcal{L}}{\partial w_1 \partial w_2} \\ \frac{\partial^2 \mathcal{L}}{\partial w_2 \partial w_1} & \frac{\partial^2 \mathcal{L}}{\partial w_2^2} \end{bmatrix}$$

**Analytical Derivation:**
Let's derive the exact mathematical formulas for these 4 entries.
Recall our first derivatives (the gradient) from the earlier tasks:

* $g_1 = \frac{\partial \mathcal{L}}{\partial w_1} = -2(a - w_1) - 4bw_1(w_2 - w_1^2)$
* $g_2 = \frac{\partial \mathcal{L}}{\partial w_2} = 2b(w_2 - w_1^2)$

Now, we take the derivative of these *again*:

1. **$H_{11}$ (Derive $g_1$ w.r.t $w_1$):**
$\frac{\partial}{\partial w_1} [-2a + 2w_1 - 4bw_1w_2 + 4bw_1^3] = 2 - 4bw_2 + 12bw_1^2$
2. **$H_{12}$ (Derive $g_1$ w.r.t $w_2$):**
$\frac{\partial}{\partial w_2} [-2(a - w_1) - 4bw_1w_2 + 4bw_1^3] = -4bw_1$
3. **$H_{21}$ (Derive $g_2$ w.r.t $w_1$):**
$\frac{\partial}{\partial w_1} [2bw_2 - 2bw_1^2] = -4bw_1$ *(Note: $H_{12}$ and $H_{21}$ are always equal for continuous functions!)*
4. **$H_{22}$ (Derive $g_2$ w.r.t $w_2$):**
$\frac{\partial}{\partial w_2} [2bw_2 - 2bw_1^2] = 2b$

Complete the cell below to return this matrix:

In [12]:
import jax

hess_rosenbrock_2D = jax.hessian(rosenbrock_2D, argnums=0)

In [13]:
import numpy as np

# Test 1: Known Hessian at [1, 1] for standard parameters a=1, b=100
expected_hessian = np.array([
    [802.0, -400.0],
    [-400.0, 200.0]
])
assert np.allclose(hess_rosenbrock_2D(np.ones(2)), expected_hessian)

### END TESTS

### The Concept: Newton's Method as a Root Finder

This task highlights a beautiful connection between optimization and root-finding.

You likely know Newton's Method from single-variable calculus as a way to find the *roots* of a function (where $f(x) = 0$). The update rule is:


$$x_{next} = x - \frac{f(x)}{f'(x)}$$

**How do we use this for optimization?**
In optimization, we are not looking for where the loss function is $0$. We are looking for where the *slope* (the gradient) is $0$.

Therefore, to minimize a function $\mathcal{L}(\mathbf{w})$, we apply Newton's root-finding method to the **gradient** $\nabla \mathcal{L}(\mathbf{w})$.
If the gradient is our "function," then we need the derivative of the gradient to act as our denominator. The derivative of a gradient vector is the **Hessian matrix**!

Because we are working with matrices, we cannot simply "divide" by the Hessian. Instead, we multiply by its inverse. The multidimensional Newton optimization step becomes:


$$\mathbf{w}_{next} = \mathbf{w} - H(\mathbf{w})^{-1} \nabla \mathcal{L}(\mathbf{w})$$


### Why is Newton's Method powerful?

Unlike Gradient Descent (which only knows the slope and blindly steps downhill), Newton's Method uses the Hessian to understand the *curvature* (the bowl shape) of the landscape. Because it knows the exact shape of the bowl, it can often jump directly to the bottom in just a few steps, completely bypassing the zigzagging problem of the Rosenbrock valley!

Now apply `newton_opt` to the problem of minimizing the Rosenbrock function:

In [14]:
def newton_opt(f, df, w0, max_iter=100, tol=1e-6):
    """
    Minimizes a function using Newton's Method.
    
    Parameters:
    f        : Function that returns the gradient vector (1D numpy array).
    df       : Function that returns the Hessian matrix (2D numpy array).
    w0       : Initial guess for the parameters (1D numpy array).
    max_iter : Maximum number of iterations to perform.
    tol      : Tolerance for convergence; stops when the step size is tiny.
    """
    # Ensure w is a float array and copy it so we don't modify the original
    w = np.array(w0, dtype=float).copy()
    
    for k in range(max_iter):
        # 1. Calculate the gradient (f) and Hessian (df) at the current position
        grad = f(w)
        hessian = df(w)
        
        # 2. Solve the linear system: H * step = -grad
        # This is numerically safer and faster than w = w - inv(H) * grad
        try:
            step = np.linalg.solve(hessian, -grad)
        except np.linalg.LinAlgError:
            print(f"Warning: Singular Hessian encountered at iteration {k}. Optimization stopped.")
            break
            
        # 3. Update the weights
        w = w + step
        
        # 4. Check for convergence (if our step size is infinitesimally small)
        step_size = np.linalg.norm(step)
        if step_size < tol:
            print(f"Newton's Method converged after {k+1} iterations!")
            break
            
    return w

In [15]:
## below, override `m_rb` to a suitable function, such that Newton's method optimizes the
## Rosenbrock function with parameters `a = 2, b = 100`.
## `m_rb` is given to `newton_opt` as the first argument.

### BEGIN SOLUTION
# The "function" we want to find the root of is the GRADIENT
m_rb = lambda w: dw_rosenbrock_2D(w, a=2.0, b=100.0)
### END SOLUTION


## below, set `dw_m_rb` to a suitable function that is used as the second argument for `newton_opt`:

### BEGIN SOLUTION
# The "derivative" of the gradient is the HESSIAN matrix
dw_m_rb = lambda w: hess_rosenbrock_2D(w, a=2.0, b=100.0)
### END SOLUTION


## let's actually call the algorithm:
w_opt_newton = newton_opt(m_rb, dw_m_rb, w0_rb)

# Optional: Print the result
print(f"Optimal weights found by Newton's Method: {w_opt_newton}")

Newton's Method converged after 7 iterations!
Optimal weights found by Newton's Method: [2.00000002 3.99999989]


In [16]:
# 1. Check if it is a vector (a 1D numpy array)
assert isinstance(w_opt_newton, np.ndarray) and w_opt_newton.ndim == 1

# 2. Check the length
assert len(w_opt_newton) == 2

### BEGIN TESTS
# 3. Check for approximate equality to [2, 4]
assert np.allclose(w_opt_newton, [2.0, 4.0])
print("Newton's Method tests passed successfully!")
### END TESTS

Newton's Method tests passed successfully!


Now, the solution trajectory should reach the optimum quickly:
![newton iterations](newton.png)